# 5.X — CellProfiler feature prediction: ridge probe to `cp_measure`

Per-feature ridge probe `embedding → cp_measure feature` on a held-out batch matched to the DataModule split (val_frac=0.1, seed=42): JUMP `source_4`, RxRx1 `HEPG2-07`, Rxrx3C `plate_4`. Headline metric is **median Pearson r** across `cp_measure` features, computed within the held-out batch (offset/scale-invariant). Secondary metric is median R²; the R²–r² gap quantifies cross-batch scale bias.

**The question**: do the learned embeddings linearly encode the morphological axes that a domain-standard tool (`cp_measure`) extracts from the same images? Recall and copairs ask whether same-perturbation samples cluster; this probe asks *which* biologically meaningful axes are encoded.

**Configurations**: three fine-tuned encoders (SubCell, DINO-ViT v3, OpenPhenom) × four input views (C / CD / S / SD), plus a single zero-shot OpenPhenom baseline (frozen HF trunk, no fine-tuning). Both pre- and post-Harmony embeddings evaluated.

**Zero-shot is on a different axis** to the fine-tuned grid — it has no view variant. Treat it as a pretrained baseline alongside Tier 0/1/2 baselines, not as a fifth view.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

TEXTWIDTH_PT = 397.48499  # \the\textwidth
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27
FULLWIDTH_IN = TEXTWIDTH_IN
HALFWIDTH_IN = TEXTWIDTH_IN / 2

def neurips_style():
    mpl.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'mathtext.fontset': 'dejavusans',
        'font.size': 7,
        'axes.titlesize': 7,
        'axes.labelsize': 7,
        'xtick.labelsize': 7,
        'ytick.labelsize': 7,
        'legend.fontsize': 7,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.linewidth': 0.6,
        'xtick.major.width': 0.6,
        'ytick.major.width': 0.6,
        'lines.linewidth': 1.0,
        'patch.linewidth': 0.6,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.01,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    })

neurips_style()

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CPM_PRED_DIR = Path('../04_cp_feature_prediction')
FIG_DIR = Path('.')
FIG_DIR.mkdir(exist_ok=True)

DATASETS = ['RxRx1', 'Rxrx3C', 'JUMP']
ENCODERS = ['SubCell', 'DINO', 'OpenPhenom']
VIEWS = ['C', 'CD', 'S', 'SD']

ENCODER_DISPLAY = {
    'SubCell':    'SubCell',
    'DINO':       'DINO\nViT-B/16',
    'OpenPhenom': 'OpenPhenom',
    'OpenPhenom-zs': 'OpenPhenom\n(zero-shot)',
}

DATASET_LABEL = {
    'RxRx1':  'RxRx1\n(siRNA, HEPG2)',
    'Rxrx3C': 'RxRx3-core\n(CRISPR-KO, HUVEC)',
    'JUMP':   'JUMP-CP\n(compounds, U2OS)',
}

CPM_PRED_CSV = {
    'RxRx1':  CPM_PRED_DIR / 'cp_feature_prediction_rxrx1.csv',
    'Rxrx3C': CPM_PRED_DIR / 'cp_feature_prediction_rxrx3c.csv',
    'JUMP':   CPM_PRED_DIR / 'cp_feature_prediction_jump.csv',
}

# CSV 'group' column is e.g. 'JUMP — OpenPhenom' (em-dash separator).
_DATASET_NORM = {'Rxrx1': 'RxRx1', 'Rxrx3C': 'Rxrx3C', 'JUMP': 'JUMP'}


In [ ]:
# Load all three datasets, normalise group/encoder columns, split out zero-shot
# baseline (encoder='OpenPhenom-zs') from the fine-tuned sweep.
frames = []
for ds, path in CPM_PRED_CSV.items():
    df = pd.read_csv(path)
    df['dataset'] = ds
    df['encoder'] = df['group'].str.split(' \u2014 ').str[1]
    df.loc[df['view'] == 'zero-shot', 'encoder'] = 'OpenPhenom-zs'
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
raw = raw[raw['embedding'].isin(['pre_harmony', 'post_harmony'])]

ft   = raw[raw['encoder'].isin(ENCODERS) & raw['view'].isin(VIEWS)].copy()
zs   = raw[raw['encoder'] == 'OpenPhenom-zs'].copy()

print(f'fine-tuned rows: {len(ft)}  (datasets={ft["dataset"].nunique()}, encoders={ft["encoder"].nunique()}, views={ft["view"].nunique()})')
print(f'zero-shot rows : {len(zs)}  (datasets={zs["dataset"].nunique()})')
raw.head()

In [ ]:
# Pivot to (dataset, encoder, embedding) × view → wide table with both Harmony stages.
# Mirrors evals/001_background_explotability cell 4.
wide = ft.pivot_table(
    index=['dataset', 'encoder', 'embedding'],
    columns='view',
    values='pearson_median',
).reset_index()

dataset_order = pd.Categorical(wide['dataset'], categories=DATASETS, ordered=True)
encoder_order = pd.Categorical(wide['encoder'], categories=ENCODERS, ordered=True)
wide = wide.assign(dataset=dataset_order, encoder=encoder_order)
wide = wide.sort_values(['embedding', 'dataset', 'encoder']).reset_index(drop=True)
wide


In [ ]:
from matplotlib.patches import Patch

VIEW_COLOR = {
    'C':  '#2271b2',
    'CD': '#7fceaf',
    'S':  '#e84b4b',
    'SD': '#ff913a',
}
VIEWS_ORDERED = ['C', 'CD', 'S', 'SD']

# Headline-figure-only encoder display: split OpenPhenom across two lines so the
# x-axis labels under each panel stack compactly. Other figures keep the
# single-token 'OpenPhenom' from cell 2.
ENCODER_DISPLAY_HEADLINE = {**ENCODER_DISPLAY, 'OpenPhenom': 'Open\nPhenom'}

bar_w = 1.0
group_inner_w = len(VIEWS_ORDERED) * bar_w
group_gap = 1.5
group_step = group_inner_w + group_gap

encoder_centers = np.array([
    ei * group_step + (group_inner_w - bar_w) / 2
    for ei in range(len(ENCODERS))
])

bar_x = {
    (enc, view): ei * group_step + vi * bar_w
    for ei, enc in enumerate(ENCODERS)
    for vi, view in enumerate(VIEWS_ORDERED)
}


def label_panel(ax, label, x=-0.15, y=1.2):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', ha='left')


def render_cpm_pred_figure(stage: str, out_stem: str, ymax: float | None = None):
    """Render the 3-panel paired-views figure for one Harmony stage."""
    df = wide[wide['embedding'] == stage]
    if ymax is None:
        ymax = df[VIEWS_ORDERED].values.max() * 1.05

    fig, axes = plt.subplots(1, 3, figsize=(FULLWIDTH_IN, 2))
    for ax, ds in zip(axes, DATASETS):
        sub = df[df['dataset'] == ds].set_index('encoder').loc[ENCODERS]

        for enc in ENCODERS:
            for view in VIEWS_ORDERED:
                ax.bar(
                    bar_x[(enc, view)],
                    sub.loc[enc, view],
                    width=bar_w,
                    color=VIEW_COLOR[view],
                    edgecolor='black',
                    linewidth=0.4,
                )

        all_x = [bar_x[(enc, v)] for enc in ENCODERS for v in VIEWS_ORDERED]
        ax.set_xticks(all_x)
        ax.set_xticklabels([''] * len(all_x))

        for centre, enc in zip(encoder_centers, ENCODERS):
            ax.text(
                centre, -0.10, ENCODER_DISPLAY_HEADLINE[enc],
                transform=ax.get_xaxis_transform(),
                ha='center', va='top',
            )

        ax.set_title(DATASET_LABEL[ds])
        ax.axhline(0, color='gray', linewidth=0.5)
        rightmost = (len(ENCODERS) - 1) * group_step + group_inner_w
        ax.set_xlim(-bar_w, rightmost)
        ax.set_ylim(0, ymax)

    for ax, lbl in zip(axes, 'ABC'):
        label_panel(ax, lbl)

    legend_handles = [Patch(color=VIEW_COLOR[v], label=v) for v in VIEWS_ORDERED]
    axes[2].legend(
        handles=legend_handles,
        loc='upper right',
        bbox_to_anchor=(1.0, 1.05),
        frameon=False,
        fontsize=7,
        handlelength=1.0,
        handleheight=1.0,
        borderpad=0.1,
        labelspacing=0.1,
    )

    stage_label = 'Post-Harmony' if stage == 'post_harmony' else 'Pre-Harmony'
    axes[0].set_ylabel(f'Median Pearson r ({stage_label})')
    fig.tight_layout()

    out_pdf = FIG_DIR / f'{out_stem}.pdf'
    fig.savefig(out_pdf)
    fig.savefig(out_pdf.with_suffix('.png'), dpi=300)
    plt.show()
    print(f'Saved: {out_pdf}')


# Shared y-axis across both stages so the two figures are directly comparable
shared_ymax = wide[VIEWS_ORDERED].values.max() * 1.05

render_cpm_pred_figure('post_harmony', 'fig_cpm_pred_post_harmony', ymax=shared_ymax)
render_cpm_pred_figure('pre_harmony',  'fig_cpm_pred_pre_harmony',  ymax=shared_ymax)




In [ ]:
# Per-cp_measure-group bar plot, 3 columns x 4 rows (datasets x views).
# Each row uses its view's colour (matching the headline figure); the three
# encoders within a panel are differentiated by hatch pattern.

GROUP_COLS = ['pearson_intensity', 'pearson_texture', 'pearson_granularity',
              'pearson_radialdistribution', 'pearson_areashape', 'pearson_zernike']
GROUP_LABELS = ['Intensity', 'Texture', 'Granularity', 'Radial dist.',
                'Area shape', 'Zernike']

# View colour reused from the headline-figure cell (VIEW_COLOR is in scope).
ROW_VIEWS = ['C', 'CD', 'S', 'SD']

ENCODER_HATCH = {
    'SubCell':    '/////',
    'DINO':       '',
    'OpenPhenom': '.....',
}

post_g = ft[ft['embedding'] == 'post_harmony'].copy()
shared_grp_ymax = post_g[GROUP_COLS].values.max() * 1.05

g_bar_w = 1.0
g_inner_w = len(ENCODERS) * g_bar_w
g_gap = 1.5
g_step = g_inner_w + g_gap
group_centers = np.array([
    gi * g_step + (g_inner_w - g_bar_w) / 2
    for gi in range(len(GROUP_LABELS))
])
bar_x_grp = {
    (gi, ei): gi * g_step + ei * g_bar_w
    for gi in range(len(GROUP_LABELS))
    for ei in range(len(ENCODERS))
}
rightmost = (len(GROUP_LABELS) - 1) * g_step + g_inner_w

fig, axes = plt.subplots(
    len(ROW_VIEWS), len(DATASETS),
    figsize=(FULLWIDTH_IN, 5.8),
    sharey=True, sharex=True,
)

for r, view in enumerate(ROW_VIEWS):
    color = VIEW_COLOR[view]
    for c, ds in enumerate(DATASETS):
        ax = axes[r, c]
        sub = (post_g[(post_g['dataset'] == ds) & (post_g['view'] == view)]
                   .set_index('encoder').loc[ENCODERS])
        for gi, gcol in enumerate(GROUP_COLS):
            for ei, enc in enumerate(ENCODERS):
                ax.bar(
                    bar_x_grp[(gi, ei)],
                    sub.loc[enc, gcol],
                    width=g_bar_w,
                    color=color,
                    edgecolor='black',
                    linewidth=0.8,
                    hatch=ENCODER_HATCH[enc],
                )
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.set_xlim(-g_bar_w, rightmost)
        ax.set_ylim(0, shared_grp_ymax)

        if r == len(ROW_VIEWS) - 1:
            ax.set_xticks(group_centers)
            ax.set_xticklabels(GROUP_LABELS, rotation=45, ha='right')
        else:
            ax.set_xticks(group_centers)
            ax.set_xticklabels([''] * len(group_centers))

        if r == 0:
            ax.set_title(DATASET_LABEL[ds])

        if c == 0:
            ax.set_ylabel(view, rotation=0, ha='right', va='center',
                          labelpad=18, fontweight='bold')

fig.supylabel('Median Pearson r (Post-Harmony)', x=0.02)

# Legend: hatch indicates encoder. Neutral grey fill so the legend reads as
# "the hatch is what changes between the three bars in each group".
legend_handles = [
    Patch(facecolor='#dddddd', edgecolor='black', hatch=ENCODER_HATCH[e],
          label=ENCODER_DISPLAY[e].replace(chr(10), ' '))
    for e in ENCODERS
]
axes[0, -1].legend(handles=legend_handles, loc='upper right', frameon=False,
                   fontsize=7, handlelength=1.5, handleheight=1.0,
                   borderpad=0.1, labelspacing=0.2)

fig.tight_layout(rect=[0.02, 0, 1, 1])
out_pdf = FIG_DIR / 'fig_cpm_pred_groups_post_harmony.pdf'
fig.savefig(out_pdf)
fig.savefig(out_pdf.with_suffix('.png'), dpi=300)
plt.show()
print(f'Saved: {out_pdf}')


In [ ]:
# Companion to the previous per-group figure: 3 cols (datasets) x 3 rows (encoders).
# Within each panel, 6 cp_measure groups x 4 view bars per group, coloured by view
# (no hatch needed — encoder is now the row dimension, view is the only within-panel
# axis). Reads the "encoder x dataset" interaction as a 3x3 cell.

GROUP_COLS_E = ['pearson_intensity', 'pearson_texture', 'pearson_granularity',
                'pearson_radialdistribution', 'pearson_areashape', 'pearson_zernike']
GROUP_LABELS_E = ['Intensity', 'Texture', 'Granularity', 'Radial dist.',
                  'Area shape', 'Zernike']

# VIEW_COLOR and VIEWS_ORDERED come from the headline-figure cell.
post_e = ft[ft['embedding'] == 'post_harmony'].copy()
shared_grp_ymax_e = post_e[GROUP_COLS_E].values.max() * 1.05

e_bar_w = 1.0
e_inner_w = len(VIEWS_ORDERED) * e_bar_w
e_gap = 1.5
e_step = e_inner_w + e_gap
group_centers_e = np.array([
    gi * e_step + (e_inner_w - e_bar_w) / 2
    for gi in range(len(GROUP_LABELS_E))
])
bar_x_e = {
    (gi, vi): gi * e_step + vi * e_bar_w
    for gi in range(len(GROUP_LABELS_E))
    for vi in range(len(VIEWS_ORDERED))
}
rightmost_e = (len(GROUP_LABELS_E) - 1) * e_step + e_inner_w

fig, axes = plt.subplots(
    len(ENCODERS), len(DATASETS),
    figsize=(FULLWIDTH_IN, 4.6),
    sharey=True, sharex=True,
)

for r, enc in enumerate(ENCODERS):
    for c, ds in enumerate(DATASETS):
        ax = axes[r, c]
        sub = (post_e[(post_e['dataset'] == ds) & (post_e['encoder'] == enc)]
                   .set_index('view').loc[VIEWS_ORDERED])
        for gi, gcol in enumerate(GROUP_COLS_E):
            for vi, view in enumerate(VIEWS_ORDERED):
                ax.bar(
                    bar_x_e[(gi, vi)],
                    sub.loc[view, gcol],
                    width=e_bar_w,
                    color=VIEW_COLOR[view],
                    edgecolor='black',
                    linewidth=0.4,
                )
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.set_xlim(-e_bar_w, rightmost_e)
        ax.set_ylim(0, shared_grp_ymax_e)

        if r == len(ENCODERS) - 1:
            ax.set_xticks(group_centers_e)
            ax.set_xticklabels(GROUP_LABELS_E, rotation=45, ha='right')
        else:
            ax.set_xticks(group_centers_e)
            ax.set_xticklabels([''] * len(group_centers_e))

        if r == 0:
            ax.set_title(DATASET_LABEL[ds])

        if c == 0:
            ax.set_ylabel(ENCODER_DISPLAY[enc].replace(chr(10), ' '),
                          rotation=90, ha='center', va='center',
                          labelpad=10, fontweight='bold')

fig.supylabel('Median Pearson r (Post-Harmony)', x=0.02)

legend_handles = [Patch(color=VIEW_COLOR[v], label=v) for v in VIEWS_ORDERED]
axes[0, -1].legend(handles=legend_handles, loc='upper right', frameon=False,
                   fontsize=7, handlelength=1.0, handleheight=1.0,
                   borderpad=0.1, labelspacing=0.1)

fig.tight_layout(rect=[0.02, 0, 1, 1])
out_pdf = FIG_DIR / 'fig_cpm_pred_groups_by_encoder_post_harmony.pdf'
fig.savefig(out_pdf)
fig.savefig(out_pdf.with_suffix('.png'), dpi=300)
plt.show()
print(f'Saved: {out_pdf}')



In [ ]:
# Best config per dataset (max median Pearson r among fine-tuned, post-Harmony).
# 1. encoder × view sweep, one block per dataset
# 2. per-cp_measure-group breakdown for best config per dataset
post = ft[ft['embedding'] == 'post_harmony']
best = post.loc[post.groupby('dataset')['pearson_median'].idxmax(),
                 ['dataset', 'encoder', 'view', 'pearson_median', 'pearson_frac_gt_0p3', 'r2_median']]
best


## §5.X prose draft (skeleton)

Placeholder. Populate after inspecting numbers.

Likely beats:
- **Encoder × dataset interaction**: OpenPhenom dominates Rxrx3C across every view; on JUMP and Rxrx1 the three fine-tuned encoders are within ~0.03–0.04 of each other. Quantify the OpenPhenom lead on Rxrx3C.
- **View interaction inverts on Rxrx1**: S/SD beats C/CD on JUMP and Rxrx3C consistently (~+0.05); on Rxrx1, C/CD beats S for DINO and OpenPhenom. Frame as evidence that segmentation views help most when batch heterogeneity is high.
- **Fine-tuning lift over the pretrained baseline is small everywhere except Rxrx3C**: report Δ (best fine-tuned − zero-shot) per dataset. Only Rxrx3C’s lift exceeds plausible single-batch noise.
- **Harmony's effect splits by held-out provenance**: cross-lab JUMP → large R² lift, no r change; same-study Rxrx3C → numerical no-op; same-study Rxrx1 → small r lift, mild R² hurt for OpenPhenom (over-correction signal).
- **Predictability hierarchy across `cp_measure` groups is universal**: Intensity ≳ Texture > Granularity ≈ RadialDistribution >> Zernike >> AreaShape. Note AreaShape outliers where zero-shot exceeds fine-tuned (suggests the contrastive objective squeezes out shape signal that the pretrained encoder retains).